<a href="https://colab.research.google.com/github/bochendong/AgentUniverse/blob/main/active_zoom_base_settings_85%25.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# train_active_zoom_cifar10_stronger.py
# A stronger, still-minimal "decision-chain zoom" classifier for CIFAR-10.
# Key fixes vs the default version:
# 1) Better global view (32x32 instead of 16x16) so the "scan whole image" stage is not too blurry.
# 2) Box/scale embedding added to each patch token (like positional encoding for variable-size crops).
# 3) Compute penalty warmup (lambda_cost ramps up later instead of from epoch 1).
# 4) Policy warm-start: first few epochs use a fixed, sensible crop (center + large scale), so patch encoder learns.

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


# -------------------------
# Differentiable crop via grid_sample (STN-style)
# -------------------------
def crop_glimpse(img, center_xy, scale, out_size=32):
    """
    img: (B, C, H, W)
    center_xy: (B, 2) in [-1, 1] normalized coords (x,y)
    scale: (B, 1) in (0, 1], fraction of full image width/height to crop
           e.g., scale=0.5 means crop half-size region.
    out: (B, C, out_size, out_size)
    """
    B, C, H, W = img.shape
    device = img.device

    lin = torch.linspace(-1, 1, out_size, device=device)
    yy, xx = torch.meshgrid(lin, lin, indexing="ij")
    base_grid = torch.stack([xx, yy], dim=-1)[None].repeat(B, 1, 1, 1)  # (B, out, out, 2)

    s = scale.view(B, 1, 1, 1).clamp(0.05, 1.0)
    grid = base_grid * s

    c = center_xy.view(B, 1, 1, 2).clamp(-1.0, 1.0)
    grid = grid + c

    patch = F.grid_sample(img, grid, mode="bilinear", padding_mode="zeros", align_corners=True)
    return patch


# -------------------------
# Small CNN encoder -> feature vector
# -------------------------
class SmallEncoder(nn.Module):
    def __init__(self, in_ch=3, feat_dim=256, base=64):
        super().__init__()
        # Slightly stronger than the original but still lightweight.
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, base, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(base, base, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # /2

            nn.Conv2d(base, base * 2, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(base * 2, base * 2, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # /4

            nn.Conv2d(base * 2, base * 4, 3, padding=1), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Linear(base * 4, feat_dim)

    def forward(self, x):
        h = self.net(x).flatten(1)
        return self.proj(h)


# -------------------------
# Active Zoom Classifier
# -------------------------
class ActiveZoomClassifier(nn.Module):
    def __init__(self, num_classes=10, feat_dim=256, hidden_dim=256, T=3, patch_out=32):
        super().__init__()
        self.T = T
        self.patch_out = patch_out

        self.global_enc = SmallEncoder(in_ch=3, feat_dim=feat_dim, base=64)
        self.patch_enc = SmallEncoder(in_ch=3, feat_dim=feat_dim, base=64)

        self.state_upd = nn.GRUCell(input_size=feat_dim, hidden_size=hidden_dim)

        self.policy = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, 4)  # x, y, logit_scale, logit_stop
        )

        # NEW: box/scale embedding (cx, cy, s) -> feat_dim
        self.box_embed = nn.Sequential(
            nn.Linear(3, feat_dim),
            nn.ReLU(inplace=True),
            nn.Linear(feat_dim, feat_dim),
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim + feat_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x, force_center=False):
        """
        x: (B,3,32,32) normalized
        force_center: if True, use a fixed sensible crop policy (warm-start)
        Returns:
          logits: (B,10)
          stop_probs: (B,T)
          centers: (B,T,2)
          scales: (B,T,1)
        """
        B = x.size(0)

        # "Zoom space": make it larger than CIFAR's 32x32
        x_hi = F.interpolate(x, size=(64, 64), mode="bilinear", align_corners=False)

        # FIX 1: better global glimpse (32x32, not 16x16)
        x_glb = F.interpolate(x_hi, size=(32, 32), mode="bilinear", align_corners=False)
        gfeat = self.global_enc(x_glb)

        # Init hidden state
        h = torch.zeros(B, self.state_upd.hidden_size, device=x.device)
        h = self.state_upd(gfeat, h)

        stop_probs = []
        centers = []
        scales = []

        cont = torch.ones(B, 1, device=x.device)  # soft continuation gate

        for _ in range(self.T):
            if force_center:
                cx = torch.zeros(B, 1, device=x.device)
                cy = torch.zeros(B, 1, device=x.device)
                s = torch.full((B, 1), 0.70, device=x.device)  # large FoV
                stop = torch.zeros(B, 1, device=x.device)      # don't stop during warm-start
            else:
                out = self.policy(h)
                cx = torch.tanh(out[:, 0:1])
                cy = torch.tanh(out[:, 1:2])
                s = 0.15 + 0.85 * torch.sigmoid(out[:, 2:3])   # (0.15, 1.0]
                stop = torch.sigmoid(out[:, 3:4])

            cxy = torch.cat([cx, cy], dim=1)  # (B,2)
            patch = crop_glimpse(x_hi, cxy, s, out_size=self.patch_out)
            pfeat = self.patch_enc(patch)

            # FIX 2: inject location+scale info (positional encoding for crops)
            box = torch.cat([cxy, s], dim=1)         # (B,3)
            pfeat = pfeat + self.box_embed(box)

            # soft stopping gate
            pfeat = pfeat * cont
            h = self.state_upd(pfeat, h)

            stop_probs.append(stop.squeeze(1))
            centers.append(cxy)
            scales.append(s)

            cont = cont * (1.0 - stop)

        stop_probs = torch.stack(stop_probs, dim=1)
        centers = torch.stack(centers, dim=1)
        scales = torch.stack(scales, dim=1)

        logits = self.classifier(torch.cat([h, gfeat], dim=1))
        return logits, stop_probs, centers, scales


# -------------------------
# Training helpers
# -------------------------
def expected_steps_from_stop_probs(stop_probs):
    """
    stop_probs: (B,T) in [0,1]
    Return: (B,) expected number of steps used (soft).
    """
    B, T = stop_probs.shape
    cont = torch.ones(B, device=stop_probs.device)
    exp_steps = torch.zeros(B, device=stop_probs.device)
    for t in range(T):
        exp_steps += cont
        cont = cont * (1.0 - stop_probs[:, t])
    return exp_steps


def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(0)

    # CIFAR-10 transforms
    tf_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2023, 0.1994, 0.2010)),
    ])
    tf_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2023, 0.1994, 0.2010)),
    ])

    train_set = datasets.CIFAR10(root="./data", train=True, download=True, transform=tf_train)
    test_set = datasets.CIFAR10(root="./data", train=False, download=True, transform=tf_test)

    train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

    model = ActiveZoomClassifier(num_classes=10, feat_dim=256, hidden_dim=256, T=3, patch_out=32).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)

    # FIX 3: compute penalty warmup/ramp
    max_lambda_cost = 0.05
    warmup_epochs_cost = 20
    ramp_epochs_cost = 20

    # FIX 4: policy warm-start
    warmstart_epochs_policy = 10

    def current_lambda_cost(epoch):
        if epoch <= warmup_epochs_cost:
            return 0.0
        t = min(epoch - warmup_epochs_cost, ramp_epochs_cost) / ramp_epochs_cost
        return max_lambda_cost * t

    @torch.no_grad()
    def eval_acc():
        model.eval()
        correct, total = 0, 0
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            logits, _, _, _ = model(x, force_center=False)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.numel()
        model.train()
        return correct / total

    for epoch in range(1, 101):
        model.train()
        lam = current_lambda_cost(epoch)
        force_center = (epoch <= warmstart_epochs_policy)

        total_loss = 0.0
        total_cls = 0.0
        total_cost = 0.0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            logits, stop_probs, _, _ = model(x, force_center=force_center)
            loss_cls = F.cross_entropy(logits, y)

            exp_steps = expected_steps_from_stop_probs(stop_probs).mean()
            loss = loss_cls + lam * exp_steps

            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total_loss += loss.item()
            total_cls += loss_cls.item()
            total_cost += exp_steps.item()

        acc = eval_acc()
        print(
            f"Epoch {epoch:03d} | "
            f"loss={total_loss/len(train_loader):.4f} | "
            f"cls={total_cls/len(train_loader):.4f} | "
            f"exp_steps={total_cost/len(train_loader):.2f} | "
            f"lambda={lam:.4f} | "
            f"warmstart={int(force_center)} | "
            f"test_acc={acc*100:.2f}%"
        )

    print("Done.")


if __name__ == "__main__":
    main()


100%|██████████| 170M/170M [00:13<00:00, 12.8MB/s]


Epoch 001 | loss=1.8569 | cls=1.8569 | exp_steps=3.00 | lambda=0.0000 | warmstart=1 | test_acc=34.51%
Epoch 002 | loss=1.5916 | cls=1.5916 | exp_steps=3.00 | lambda=0.0000 | warmstart=1 | test_acc=38.82%
Epoch 003 | loss=1.4363 | cls=1.4363 | exp_steps=3.00 | lambda=0.0000 | warmstart=1 | test_acc=48.35%
Epoch 004 | loss=1.3099 | cls=1.3099 | exp_steps=3.00 | lambda=0.0000 | warmstart=1 | test_acc=51.44%
Epoch 005 | loss=1.2060 | cls=1.2060 | exp_steps=3.00 | lambda=0.0000 | warmstart=1 | test_acc=57.56%
Epoch 006 | loss=1.1274 | cls=1.1274 | exp_steps=3.00 | lambda=0.0000 | warmstart=1 | test_acc=54.82%
Epoch 007 | loss=1.0599 | cls=1.0599 | exp_steps=3.00 | lambda=0.0000 | warmstart=1 | test_acc=62.24%
Epoch 008 | loss=1.0011 | cls=1.0011 | exp_steps=3.00 | lambda=0.0000 | warmstart=1 | test_acc=62.04%
Epoch 009 | loss=0.9500 | cls=0.9500 | exp_steps=3.00 | lambda=0.0000 | warmstart=1 | test_acc=64.73%
Epoch 010 | loss=0.9013 | cls=0.9013 | exp_steps=3.00 | lambda=0.0000 | warmstart=

KeyboardInterrupt: 